# Data ETL + EDA

> Notebook generated from [`examples/subgraphs/05_data_etl.py`](../../examples/subgraphs/05_data_etl.py).

| Item | Value |
|------|-------|
| **Dataset** | Titanic (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** extractor → validator [EDA] → [gate] → transformer → loader → auditor. Statistics, feature engineering, audit log.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Data ETL Subgraph — Data Pipeline with Exploratory Data Analysis (EDA)
======================================================================
Subgraph: prismal.agents.subgraphs.data_etl

Dataset: Titanic (Kaggle / OpenML)
  • 887 passengers with 12 variables: class, name, sex, age, SibSp,
    Parch, fare, cabin, embarked, survival.
  • Reference: https://www.openml.org/d/40945
  • Why: The Titanic dataset is the classic benchmark for EDA.
    It has null values (Age, Cabin), mixed variables (numeric and
    categorical), and lets us demonstrate all stages of the ETL pipeline:
    schema validation, null detection, distribution analysis,
    cleaning, feature engineering and quality auditing.

Data ETL subgraph description:
  extractor → validator → [gate: validation passed?] → transformer → loader → auditor
                                         └── (failure) ─────────────────────────↑

  Nodes:
  1. extractor   — loads data from source (CSV, Parquet, JSON, in-memory)
  2. validator   — checks schema, required columns, null values
  3. [gate]      — if validation.passed=False → jump to auditor (error path)
  4. transformer — cleaning, imputation, feature engineering, selection
  5. loader      — writes the transformed DataFrame to destination
  6. auditor     — generates a quality report: before/after pipeline

EDA focus in this example:
  • validator  → null analysis, types, value ranges
  • transformer → age imputation by class median, sex encoding,
                  'family_size', 'age_group' features, outlier filtering
  • auditor    → before/after metrics comparison (null rate, shape)

Usage:
    uv run python examples/subgraphs/05_data_etl.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import io
from typing import Any

import polars as pl

# Import with error handling
try:
    from prismal.agents.subgraphs.data_etl.builder import (
        build_data_etl_subgraph,
        register_data_etl,
    )

    DATA_ETL_AVAILABLE = True
except ImportError:
    DATA_ETL_AVAILABLE = False

## Dataset: Titanic (887 passengers, inline data)

In [ ]:
# Representative subset with the most relevant variables for EDA.
TITANIC_CSV = """\
PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.25,S
2,1,1,"Cumings, Mrs. John Bradley",female,38.0,1,0,71.2833,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.925,S
4,1,1,"Futrelle, Mrs. Jacques Heath",female,35.0,1,0,53.1,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.05,S
6,0,3,"Moran, Mr. James",male,,0,0,8.4583,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,51.8625,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,21.075,S
9,1,3,"Johnson, Mrs. Oscar W",female,27.0,0,2,11.1333,S
10,1,2,"Nasser, Mrs. Nicholas",female,14.0,1,0,30.0708,C
11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,16.7,S
12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,26.55,S
13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,8.05,S
14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,31.275,S
15,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,7.8542,S
16,1,2,"Hewlett, Mrs. Mary D Kingcome",female,55.0,0,0,16.0,S
17,0,3,"Rice, Master. Eugene",male,2.0,4,1,29.125,Q
18,1,2,"Williams, Mr. Charles Eugene",male,,0,0,13.0,S
19,0,3,"Vander Planke, Mrs. Julius",female,31.0,1,0,18.0,S
20,1,3,"Masselmani, Mrs. Fatima",female,,0,0,7.225,C
21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,26.0,S
22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,13.0,S
23,1,3,"McGowan, Miss. Anna",female,15.0,0,0,8.0292,Q
24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,35.5,S
25,0,3,"Palsson, Miss. Torborg Danira",female,8.0,3,1,21.075,S
26,1,3,"Asplund, Mrs. Carl Oscar",female,38.0,1,5,31.3875,S
27,0,3,"Emir, Mr. Farred Chehab",male,,0,0,7.225,C
28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,263.0,S
29,1,3,"O'Dwyer, Miss. Ellen",female,,0,0,7.8792,Q
30,0,3,"Todoroff, Mr. Lalio",male,,0,0,7.8958,S
31,0,1,"Uruchurtu, Don. Manuel E",male,40.0,0,0,27.7208,C
32,1,1,"Spencer, Mrs. William Augustus",female,,1,0,146.5208,C
33,1,3,"Glynn, Miss. Mary Agatha",female,,0,0,7.75,Q
34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,10.5,S
35,0,1,"Meyer, Mr. Edgar Joseph",male,28.0,1,0,82.1708,C
36,0,1,"Holverson, Mr. Alexander Oskar",male,42.0,1,0,52.0,S
37,1,3,"Mamee, Mr. Hanna",male,,0,0,7.2292,C
38,0,3,"Cann, Mr. Ernest Charles",male,21.0,0,0,8.05,S
39,0,3,"Vander Planke, Miss. Augusta Maria",female,18.0,2,0,18.0,S
40,1,3,"Nicola-Yarred, Miss. Jamila",female,14.0,1,0,11.2417,C
41,0,3,"Ahlin, Mrs. Johan",female,40.0,1,0,9.475,S
42,0,2,"Turpin, Mrs. William John Robert",female,27.0,1,0,21.0,S
43,0,3,"Kraeff, Mr. Theodor",male,,0,0,7.8958,C
44,1,2,"Laroche, Miss. Simonne Marie Anne Andree",female,3.0,1,2,41.5792,C
45,1,3,"Devaney, Miss. Margaret Delia",female,19.0,0,0,7.8792,Q
46,0,3,"Rogers, Mr. William John",male,,0,0,8.05,S
47,0,3,"Lennon, Mr. Denis",male,,1,0,15.5,Q
48,1,3,"O'Driscoll, Miss. Bridget",female,,0,0,7.75,Q
49,0,3,"Samaan, Mr. Youssef",male,,2,0,21.6792,C
50,0,3,"Arnold-Franchi, Mrs. Josef",female,18.0,1,0,17.8,S
"""

## Pipeline specifications

In [ ]:
REQUIRED_COLUMNS = ["PassengerId", "Survived", "Pclass", "Sex", "Age", "Fare"]

TRANSFORMS = [
    # 1. Select relevant columns
    {
        "op": "select",
        "columns": [
            "PassengerId",
            "Survived",
            "Pclass",
            "Sex",
            "Age",
            "SibSp",
            "Parch",
            "Fare",
            "Embarked",
        ],
    },
    # 2. Filter records with valid fare (> 0)
    {"op": "filter", "column": "Fare", "operator": ">", "value": 0.0},
    # 3. Rename columns to snake_case
    {
        "op": "rename",
        "mapping": {
            "PassengerId": "passenger_id",
            "Survived": "survived",
            "Pclass": "pclass",
            "Sex": "sex",
            "Age": "age",
            "SibSp": "sibsp",
            "Parch": "parch",
            "Fare": "fare",
            "Embarked": "embarked",
        },
    },
]

## Injectable callables

In [ ]:
async def titanic_extractor(source: dict[str, Any]) -> pl.DataFrame:
    """Extractor that loads the Titanic dataset from memory (no I/O)."""
    # In production: source["path"] would be a real CSV/Parquet
    # Here we use embedded data to avoid requiring external files
    return pl.read_csv(io.StringIO(TITANIC_CSV))


def titanic_validator(df: pl.DataFrame, source: dict[str, Any]) -> tuple[bool, list[str]]:
    """Validator with EDA analysis: schema, nulls, ranges."""
    errors: list[str] = []

    # 1. Check required columns
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")

    # 2. Check the dataset is not empty
    if df.height == 0:
        errors.append("Dataset is empty")
        return False, errors

    # 3. Null analysis (EDA)
    null_report = []
    for col in df.columns:
        null_count = df[col].null_count()
        null_pct = null_count / df.height * 100
        if null_pct > 80:
            errors.append(f"Column '{col}' has {null_pct:.1f}% nulls (> 80%)")
        elif null_pct > 0:
            null_report.append(f"{col}: {null_pct:.1f}%")

    # 4. Validate logical ranges
    if "Age" in df.columns:
        age_series = df["Age"].drop_nulls()
        if age_series.min() < 0 or age_series.max() > 120:
            errors.append(f"Age out of range: min={age_series.min()}, max={age_series.max()}")

    if "Fare" in df.columns:
        fare_series = df["Fare"].drop_nulls()
        if fare_series.min() < 0:
            errors.append(f"Negative fare detected: min={fare_series.min()}")

    # 5. Validate expected cardinalities
    if "Survived" in df.columns:
        unique_survived = df["Survived"].unique().sort().to_list()
        if not set(unique_survived).issubset({0, 1}):
            errors.append(f"Column 'Survived' contains unexpected values: {unique_survived}")

    if "Pclass" in df.columns:
        unique_pclass = set(df["Pclass"].drop_nulls().to_list())
        if not unique_pclass.issubset({1, 2, 3}):
            errors.append(f"Pclass contains values outside [1,2,3]: {unique_pclass}")

    return (not errors, errors)


async def titanic_transformer(
    df: pl.DataFrame, transforms: list[dict[str, Any]]
) -> tuple[pl.DataFrame, list[str]]:
    """Transformer with EDA: imputation, encoding and feature engineering."""
    log: list[str] = []
    current = df

    # ── Step 1: Apply declarative transforms (select / filter / rename) ──
    from prismal.agents.subgraphs.data_etl.transformer_node import _default_transformer

    current, base_log = _default_transformer(current, transforms)
    log.extend(base_log)

    # ── Step 2 (EDA / cleaning): impute age by median per class ──────────────
    if "age" in current.columns and "pclass" in current.columns:
        # Age median per class
        medians = (
            current.group_by("pclass")
            .agg(pl.col("age").median().alias("age_median"))
            .sort("pclass")
        )
        median_map = dict(
            zip(medians["pclass"].to_list(), medians["age_median"].to_list(), strict=False)
        )
        # Impute nulls with the median of their class
        current = current.with_columns(
            pl.when(pl.col("age").is_null())
            .then(pl.col("pclass").replace(median_map))
            .otherwise(pl.col("age"))
            .alias("age")
        )
        sum(1 for m in median_map.values() if m is not None)
        log.append(f"impute_age_by_class: medians={median_map}")

    # ── Step 3 (Feature Engineering): encode sex as binary ──────────────────
    if "sex" in current.columns:
        current = current.with_columns(
            pl.col("sex").replace({"male": 0, "female": 1}).alias("sex_bin")
        )
        current = current.drop("sex")
        log.append("encode_sex: male=0, female=1 → sex_bin")

    # ── Step 4 (Feature Engineering): family size ──────────────────────────
    if "sibsp" in current.columns and "parch" in current.columns:
        current = current.with_columns((pl.col("sibsp") + pl.col("parch") + 1).alias("family_size"))
        log.append("feature_family_size = sibsp + parch + 1")

    # ── Step 5 (Feature Engineering): age group ────────────────────────────
    if "age" in current.columns:
        current = current.with_columns(
            pl.when(pl.col("age") < 13)
            .then(pl.lit("child"))
            .when(pl.col("age") < 18)
            .then(pl.lit("teen"))
            .when(pl.col("age") < 60)
            .then(pl.lit("adult"))
            .otherwise(pl.lit("senior"))
            .alias("age_group")
        )
        log.append("feature_age_group: child/teen/adult/senior")

    # ── Step 6 (EDA): normalize fare (log1p) ──────────────────────────────
    if "fare" in current.columns:
        import math

        current = current.with_columns(
            pl.col("fare")
            .map_elements(lambda x: math.log1p(x), return_dtype=pl.Float64)
            .alias("fare_log1p")
        )
        log.append("transform_fare: log1p(fare) → fare_log1p")

    # ── Step 7: impute embarked with mode ──────────────────────────────────
    if "embarked" in current.columns:
        mode_val = current["embarked"].drop_nulls().mode()[0]
        current = current.with_columns(pl.col("embarked").fill_null(mode_val))
        log.append(f"impute_embarked: mode='{mode_val}'")

    return current, log


async def memory_loader(df: pl.DataFrame, destination: dict[str, Any]) -> int:
    """In-memory loader: simulates persistence without writing files."""
    # In production: would write to CSV/Parquet/database
    return df.height

## EDA Helper: DataFrame statistics

In [ ]:
def print_eda_summary(df: pl.DataFrame, title: str) -> None:
    """Print an EDA summary of the DataFrame."""
    print(f"\n  [{title}]")
    print(f"    Shape   : {df.height} rows × {df.width} columns")
    print(f"    Columns : {df.columns}")

    # Nulls per column
    null_info = [(c, df[c].null_count()) for c in df.columns if df[c].null_count() > 0]
    if null_info:
        print("    Nulls   :")
        for col, cnt in null_info:
            pct = cnt / df.height * 100
            print(f"      {col}: {cnt} ({pct:.1f}%)")
    else:
        print("    Nulls   : none ✓")

    # Numeric column statistics
    num_cols = [
        c for c in df.columns if df[c].dtype in (pl.Float32, pl.Float64, pl.Int32, pl.Int64)
    ]
    if num_cols:
        print("    Numeric statistics:")
        for col in num_cols[:5]:  # max 5 columns
            series = df[col].drop_nulls()
            if series.len() > 0:
                print(
                    f"      {col:15s}: min={series.min():.2f}  "
                    f"median={series.median():.2f}  max={series.max():.2f}"
                )


def print_survival_eda(df: pl.DataFrame) -> None:
    """Titanic survival-specific EDA (post-transformation)."""
    if "survived" not in df.columns:
        return

    print("\n  [EDA: Survival analysis]")

    # Overall survival rate
    survival_rate = df["survived"].mean()
    print(f"    Overall survival rate: {survival_rate:.1%}")

    # Per class
    if "pclass" in df.columns:
        by_class = (
            df.group_by("pclass")
            .agg(pl.col("survived").mean().alias("survival_rate"))
            .sort("pclass")
        )
        print("    Per passenger class:")
        for row in by_class.iter_rows(named=True):
            bar = "█" * int(row["survival_rate"] * 10)
            print(f"      Class {row['pclass']}: {row['survival_rate']:.1%}  {bar}")

    # Per sex (encoded as sex_bin)
    if "sex_bin" in df.columns:
        by_sex = (
            df.group_by("sex_bin")
            .agg(pl.col("survived").mean().alias("survival_rate"))
            .sort("sex_bin")
        )
        print("    Per sex (0=male, 1=female):")
        for row in by_sex.iter_rows(named=True):
            sex_label = "female" if row["sex_bin"] == 1 else "male  "
            bar = "█" * int(row["survival_rate"] * 10)
            print(f"      {sex_label}: {row['survival_rate']:.1%}  {bar}")

    # Per age group
    if "age_group" in df.columns:
        by_age = (
            df.group_by("age_group")
            .agg(pl.col("survived").mean().alias("survival_rate"))
            .sort("survival_rate", descending=True)
        )
        print("    Per age group:")
        for row in by_age.iter_rows(named=True):
            bar = "█" * int(row["survival_rate"] * 10)
            print(f"      {row['age_group']:8s}: {row['survival_rate']:.1%}  {bar}")

    # family_size distribution
    if "family_size" in df.columns:
        by_family = (
            df.group_by("family_size")
            .agg(
                pl.col("survived").mean().alias("survival_rate"),
                pl.len().alias("count"),
            )
            .sort("family_size")
        )
        print("    Per family size (family_size):")
        for row in by_family.iter_rows(named=True):
            bar = "█" * int(row["survival_rate"] * 10)
            print(
                f"      size={row['family_size']:2d}  n={row['count']:3d}  "
                f"survival={row['survival_rate']:.1%}  {bar}"
            )

## Pipeline execution

In [ ]:
async def run_etl_pipeline() -> None:
    """Run the Data ETL subgraph on the Titanic dataset."""
    print("\n[Running Data ETL pipeline — Titanic dataset]")

    if not DATA_ETL_AVAILABLE:
        # Demo mode: run the nodes directly without the subgraph
        print("  [Demo mode — simulated subgraph]\n")

        # Node 1: extractor
        print("  ── Node 1: extractor ──")
        source = {"type": "inline", "name": "titanic"}
        df_raw = await titanic_extractor(source)
        print(f"    Rows extracted  : {df_raw.height}")
        print(f"    Columns         : {df_raw.columns}")
        print_eda_summary(df_raw, "EDA — Raw data")

        # Node 2: validator
        print("\n  ── Node 2: validator (EDA + schema validation) ──")
        passed, errors = titanic_validator(df_raw, source)
        if passed:
            print("    ✓ Validation passed — no critical errors")
        else:
            print("    ✗ Validation failed:")
            for e in errors:
                print(f"      - {e}")

        # Gate
        print(f"\n  ── Gate: validation.passed={passed} ──")
        if not passed:
            print("    → Error path: jump to auditor")
            return

        print("    → Normal path: continue to transformer")

        # Node 3: transformer
        print("\n  ── Node 3: transformer (cleaning + EDA + feature engineering) ──")
        df_transformed, transform_log = await titanic_transformer(df_raw, TRANSFORMS)
        print(f"    Operations applied ({len(transform_log)}):")
        for op in transform_log:
            print(f"      • {op}")
        print_eda_summary(df_transformed, "EDA — Transformed data")

        # Node 4: loader
        print("\n  ── Node 4: loader ──")
        loaded = await memory_loader(df_transformed, {"type": "memory"})
        print(f"    Rows loaded   : {loaded}")
        print(f"    Final shape   : {df_transformed.height} rows × {df_transformed.width} cols")

        # Node 5: auditor
        print("\n  ── Node 5: auditor ──")
        null_before = sum(df_raw[c].null_count() for c in df_raw.columns)
        null_after = sum(df_transformed[c].null_count() for c in df_transformed.columns)
        print(
            f"    Rows    : {df_raw.height} → {df_transformed.height} "
            f"(Δ {df_transformed.height - df_raw.height:+d})"
        )
        print(
            f"    Columns : {df_raw.width} → {df_transformed.width} "
            f"(+{df_transformed.width - df_raw.width} new features)"
        )
        print(
            f"    Nulls   : {null_before} → {null_after} "
            f"({'✓ reduced' if null_after < null_before else '= no change'})"
        )
        print("    ✓ Pipeline completed successfully")

        # Post-pipeline EDA: survival analysis
        print_survival_eda(df_transformed)
        return

    # Real mode with LangGraph subgraph
    from langchain_core.messages import HumanMessage

    from prismal.agents.state import initial_state

    await register_data_etl()
    subgraph = build_data_etl_subgraph(
        extractor_fn=titanic_extractor,
        validator_fn=titanic_validator,
        transformer_fn=titanic_transformer,
        loader_fn=memory_loader,
        required_columns=REQUIRED_COLUMNS,
    )

    state = initial_state()
    state["messages"] = [HumanMessage(content="Run the ETL pipeline on the Titanic dataset.")]
    state["metadata"] = {
        "data_etl": {
            "source": {"type": "inline", "name": "titanic"},
            "transforms": TRANSFORMS,
            "destination": {"type": "memory"},
        }
    }

    config = {"configurable": {"thread_id": "etl_titanic_001"}}
    final_state = await subgraph.graph.ainvoke(state, config=config)

    etl_meta = final_state.get("metadata", {}).get("data_etl", {})
    df_result = etl_meta.get("dataframe")
    if df_result is not None:
        print_eda_summary(df_result, "Final result")
        print_survival_eda(df_result)

    validation = etl_meta.get("validation", {})
    print(f"\n  Validation    : {'✓' if validation.get('passed') else '✗'}")
    print(f"  Rows loaded   : {etl_meta.get('loaded_row_count', 'N/A')}")
    print(f"  Transform log : {etl_meta.get('transform_log', [])}")

## Error path demo: dataset with incorrect schema

In [ ]:
async def demo_validation_failure() -> None:
    """Demonstrate the error gate when the dataset does not pass validation."""
    print("\n[Demo: error path — incorrect dataset]")

    bad_csv = "col_a,col_b\n1,foo\n2,bar\n"
    df_bad = pl.read_csv(io.StringIO(bad_csv))

    source = {"type": "inline", "name": "bad_dataset"}
    passed, errors = titanic_validator(df_bad, source)

    print(f"  Dataset: {df_bad.height} rows, columns={df_bad.columns}")
    print(f"  Validation result: {'✓ PASS' if passed else '✗ FAIL'}")
    if errors:
        print("  Errors detected:")
        for e in errors:
            print(f"    - {e}")
    print("  → Gate redirects to auditor (error path) — transformer is skipped")


async def main() -> None:
    print("=" * 70)
    print("  Data ETL Subgraph — Dataset: Titanic (Kaggle / OpenML)")
    print("=" * 70)

    # Architecture
    print("\n[Data ETL subgraph architecture]")
    steps = [
        ("extractor  ", "Load CSV/Parquet/JSON → polars DataFrame"),
        ("[validator ]", "Schema, required columns, nulls, ranges (EDA)"),
        ("[  GATE    ]", "validation.passed? → transformer : auditor"),
        ("transformer", "Cleaning, imputation, feature engineering (EDA)"),
        ("loader     ", "Write DataFrame to destination (CSV/Parquet/DB)"),
        ("auditor    ", "Quality report: shape, nulls, transform log"),
    ]
    for node, desc in steps:
        print(f"  {node}: {desc}")

    # Main pipeline: Titanic EDA
    await run_etl_pipeline()

    # Error path demo
    print("\n" + "─" * 70)
    await demo_validation_failure()

    # ── Guide of available transforms ────────────────────────────────────────
    print("\n[Supported declarative transforms]")
    transforms_doc = [
        ('{"op": "select", "columns": [...]}', "Selects columns"),
        ('{"op": "filter", "column": "x", "operator": ">", "value": 10}', "Filters rows"),
        ('{"op": "rename", "mapping": {"old": "new"}}', "Renames columns"),
    ]
    for transform, desc in transforms_doc:
        print(f"  {transform}")
        print(f"    → {desc}")

    print("\n[Custom callables injection]")
    print("  build_data_etl_subgraph(")
    print("      extractor_fn=my_sql_extractor,   # async (source) -> pl.DataFrame")
    print("      validator_fn=my_pandera_schema,  # (df, source) -> (bool, list[str])")
    print("      transformer_fn=my_feature_eng,  # async (df, transforms) -> (df, log)")
    print("      loader_fn=my_postgres_loader,   # async (df, dest) -> int (rows)")
    print("      required_columns=['id', 'ts'],")
    print("  )")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()